# CKD — LASSO + Rank Fusion: Primary Analysis vs. Sensitivity Analysis

This notebook runs the **same pipeline twice** on the same underlying CKD data, so the two runs can be compared directly:

- **Primary Analysis**: the full 400-row dataset, with missing values handled by median (numerical) / mode (categorical) imputation, fit on training data only within each fold.
- **Sensitivity Analysis**: only the 158 complete-case rows (zero missing values anywhere), no imputation needed at all.

**Why this comparison matters, concretely**: dropping every row with a missing value would cost 60.5% of the dataset and visibly shifts the class balance (62.5%/37.5% CKD in the full data vs. 27.2%/72.8% in the complete cases) — a strong sign the missingness isn't random. If the Primary and Sensitivity results agree, that's real evidence the imputation approach isn't distorting the conclusions. If they disagree, that's equally real evidence — and tells you exactly where to be cautious.

**What differs from `ckd_anova.ipynb`**: the mathematical operator is **LASSO** (L1-penalized Logistic Regression, absolute coefficient magnitude as the importance score) instead of ANOVA's F-statistic. `SklearnAdapter` has an explicit branch for this (`elif hasattr(self.estimator, "coef_"): values = np.abs(self.estimator.coef_)`, labelled "Linear models / LASSO" in the package source), so no adapter changes are needed — only swapping which estimator gets wrapped. Everything else — the Primary/Sensitivity split, the imputation logic, the 25-run repeated-CV stability and performance analyses, and the Wilcoxon + Holm + Cohen's d significance testing — is identical, so the two notebooks are directly comparable.

**Fusion**: `doda.fusion.RankFusion` — genuine Reciprocal Rank Fusion, `RRF(f) = 1/(k + math_rank(f)) + 1/(k + clinical_rank(f))` (Cormack, Clarke & Buettcher, 2009).

**Imbalance handling**: `class_weight="balanced"` / `scale_pos_weight` throughout, since CKD is ~63/37.

In [1]:
# =============================================================================
# STEP 1: LOAD AND CLEAN RAW DATA (shared by both analyses)
# =============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/ckd.csv")
df = df.drop(columns=["id"])

# Known data-quality issues in this exact UCI file (see 01_ckd_eda.ipynb)
categorical_cols_raw = df.select_dtypes(include="object").columns
for col in categorical_cols_raw:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Encode categorical features and target
target_column = "target"

binary_maps = {
    "rbc":   {"normal": 0, "abnormal": 1},
    "pc":    {"normal": 0, "abnormal": 1},
    "pcc":   {"notpresent": 0, "present": 1},
    "ba":    {"notpresent": 0, "present": 1},
    "htn":   {"no": 0, "yes": 1},
    "dm":    {"no": 0, "yes": 1},
    "cad":   {"no": 0, "yes": 1},
    "appet": {"poor": 0, "good": 1},
    "pe":    {"no": 0, "yes": 1},
    "ane":   {"no": 0, "yes": 1},
}
for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[target_column] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.drop(columns=["classification"])

numerical_features = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
                       "sod", "pot", "hemo", "pcv", "wc", "rc"]
categorical_features = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad",
                         "appet", "pe", "ane"]

print("=" * 70)
print("CLEANED + ENCODED DATASET")
print("=" * 70)
print(f"Shape: {df.shape}")
display(df.head())

CLEANED + ENCODED DATASET
Shape: (400, 25)


/tmp/ipykernel_103/1346706842.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_raw = df.select_dtypes(include="object").columns


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,target
0,48.0,80.0,1.020,1.0,0.0,NaN,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,0.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1


In [2]:
# =============================================================================
# STEP 2: DEFINE THE TWO DATASETS — PRIMARY (imputed) vs SENSITIVITY (complete-case)
# =============================================================================

# --- Primary: full dataset, missing values handled by imputation later ---
X_primary = df.drop(columns=[target_column])
y_primary = df[target_column]

# --- Sensitivity: complete cases only, no imputation needed ---
df_complete = df.dropna()
X_sensitivity = df_complete.drop(columns=[target_column])
y_sensitivity = df_complete[target_column]

print("=" * 70)
print("PRIMARY vs SENSITIVITY DATASET SIZES")
print("=" * 70)
print(f"Primary (full, to be imputed) : {X_primary.shape[0]} rows")
print(f"Sensitivity (complete-case)    : {X_sensitivity.shape[0]} rows "
      f"({X_sensitivity.shape[0]/X_primary.shape[0]*100:.1f}% of primary)")

print("\nClass balance comparison:")
print("Primary:")
display((y_primary.value_counts(normalize=True) * 100).round(2))
print("Sensitivity:")
display((y_sensitivity.value_counts(normalize=True) * 100).round(2))

PRIMARY vs SENSITIVITY DATASET SIZES
Primary (full, to be imputed) : 400 rows
Sensitivity (complete-case)    : 158 rows (39.5% of primary)

Class balance comparison:
Primary:


target
1    62.5
0    37.5
Name: proportion, dtype: float64

Sensitivity:


target
0    72.78
1    27.22
Name: proportion, dtype: float64

In [3]:
%pip uninstall -y doda

In [4]:
%pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

  Cloning https://github.com/anandha-3679/DODA.git to /tmp/pip-req-build-njdzi5dv


  Running command git clone --filter=blob:none --quiet https://github.com/anandha-3679/DODA.git /tmp/pip-req-build-njdzi5dv


  Resolved https://github.com/anandha-3679/DODA.git to commit bb479647305e6c7166c6d8a0c4f93d6291f36998
  Installing build dependencies: started


  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started


  Preparing metadata (pyproject.toml): finished with status 'done'


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--break-system-packages', '--no-cache-dir', 'git+https://github.com/anandha-3679/DODA.git'], returncode=0)

In [5]:
import doda

print(doda.__file__)

from doda.knowledge import JSONProvider
from doda.fusion import RankFusion

print("JSONProvider:", JSONProvider)
print("RankFusion:", RankFusion)

/usr/local/lib/python3.12/dist-packages/doda/__init__.py
JSONProvider: <class 'doda.knowledge.providers.json_provider.JSONProvider'>
RankFusion: <class 'doda.fusion.rank.RankFusion'>


In [6]:
# =============================================================================
# DODA IMPORTS
# =============================================================================

from sklearn.feature_selection import (
    SelectKBest,
    f_classif
)

from doda import DODASelector

from doda.adapters import (
    SklearnAdapter
)

from doda.knowledge.providers import (
    JSONProvider
)

from doda.fusion import (
    RankFusion
)

In [7]:
# =============================================================================
# STEP 3: SHARED HELPER FUNCTIONS
# Defined once, called twice (Primary and Sensitivity) — avoids duplicating
# ~150 lines of loop logic twice with only the input data differing.
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

from doda import DODASelector
from doda.adapters.sklearn_adapter import SklearnAdapter
from doda.knowledge import JSONProvider
from doda.fusion import RankFusion

WEIGHTS_FILE = "../../../config/clinical_weights/ckd_clinical_weights.json"


def make_models(y_train):
    """Fresh model instances, class-imbalance-aware (CKD is ~63/37)."""
    return {
        "LR": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=42, n_jobs=-1),
        "XGB": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                              scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                              random_state=42, n_jobs=-1)
    }


def impute_if_needed(X_train, X_test, do_impute):
    """Median/mode imputation fit on TRAIN only — skipped entirely for the
    complete-case (Sensitivity) dataset, since it has no missing values."""
    if not do_impute:
        return X_train.copy(), X_test.copy()

    num_cols = [c for c in numerical_features if c in X_train.columns]
    cat_cols = [c for c in categorical_features if c in X_train.columns]

    num_imp = SimpleImputer(strategy="median")
    cat_imp = SimpleImputer(strategy="most_frequent")

    X_train_i, X_test_i = X_train.copy(), X_test.copy()
    X_train_i[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test_i[num_cols] = num_imp.transform(X_test[num_cols])
    X_train_i[cat_cols] = cat_imp.fit_transform(X_train[cat_cols])
    X_test_i[cat_cols] = cat_imp.transform(X_test[cat_cols])

    return X_train_i, X_test_i


def run_stability_analysis(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV. For each K, runs BOTH LASSO and
    DODA, records selected feature sets, computes pairwise Jaccard similarity."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    all_jaccard = []

    for k in k_values:
        print(f"\n{'='*70}\n{label} STABILITY — TOP-{k}\n{'='*70}")

        for method in ["LASSO", "DODA"]:
            selected_sets = []

            for train_idx, _ in cv.split(X, y):
                X_train = X.iloc[train_idx]
                y_train = y.iloc[train_idx]
                X_train_imp, _ = impute_if_needed(X_train, X_train, do_impute)

                if method == "LASSO":
                    lasso = LogisticRegression(
                        penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
                        class_weight="balanced", random_state=42
                    )
                    lasso.fit(X_train_imp, y_train)
                    coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_imp.columns)
                    features = coefs.sort_values(ascending=False).head(k).index.tolist()
                else:
                    operator = SklearnAdapter(
                        LogisticRegression(
                            penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
                            class_weight="balanced", random_state=42
                        )
                    )
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    selector.fit(X_train_imp, y_train)
                    features = list(selector.get_selected_features())

                selected_sets.append(set(features))

            jaccard_scores = []
            for i in range(len(selected_sets)):
                for j in range(i + 1, len(selected_sets)):
                    inter = len(selected_sets[i] & selected_sets[j])
                    union = len(selected_sets[i] | selected_sets[j])
                    jaccard_scores.append(inter / union)

            for score in jaccard_scores:
                all_jaccard.append({"Top_K": k, "Method": method, "Jaccard": score})

            print(f"{method}: mean Jaccard = {np.mean(jaccard_scores):.4f} "
                  f"(std {np.std(jaccard_scores):.4f})")

    return pd.DataFrame(all_jaccard)


def run_cv_performance(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV predictive performance,
    LASSO vs DODA, across 3 models and all Top-K values."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    results = []

    for run_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute)

        for k in k_values:
            for method in ["LASSO", "DODA"]:
                if method == "LASSO":
                    lasso = LogisticRegression(
                        penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
                        class_weight="balanced", random_state=42
                    )
                    lasso.fit(X_train_imp, y_train)
                    coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_imp.columns)
                    sel_features = coefs.sort_values(ascending=False).head(k).index.tolist()
                    X_train_sel = X_train_imp[sel_features]
                    X_test_sel = X_test_imp[sel_features]
                else:
                    operator = SklearnAdapter(
                        LogisticRegression(
                            penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
                            class_weight="balanced", random_state=42
                        )
                    )
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    X_train_sel = selector.fit_transform(X_train_imp, y_train)
                    X_test_sel = selector.transform(X_test_imp)

                models = make_models(y_train)
                for model_name, model in models.items():
                    if model_name == "LR":
                        scaler = StandardScaler()
                        Xtr = scaler.fit_transform(X_train_sel)
                        Xts = scaler.transform(X_test_sel)
                    else:
                        Xtr, Xts = X_train_sel, X_test_sel

                    model.fit(Xtr, y_train)
                    y_pred = model.predict(Xts)
                    y_prob = model.predict_proba(Xts)[:, 1]

                    results.append({
                        "Run": run_id, "Top_K": k, "Method": method, "Model": model_name,
                        "Accuracy": accuracy_score(y_test, y_pred),
                        "F1": f1_score(y_test, y_pred, zero_division=0),
                        "ROC_AUC": roc_auc_score(y_test, y_prob)
                    })

        if run_id % 5 == 0:
            print(f"{label}: completed {run_id}/25 CV runs")

    return pd.DataFrame(results)


def wilcoxon_holm_test(df, group_cols, value_col="Jaccard"):
    """Paired Wilcoxon signed-rank test (LASSO vs DODA) + Cohen's d,
    with Holm correction across all comparisons in this dataframe."""
    rows = []
    for keys, group in df.groupby(group_cols):
        lasso_vals = group[group.Method == "LASSO"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "LASSO"].sort_values("Run")[value_col].values
        doda_vals = group[group.Method == "DODA"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "DODA"].sort_values("Run")[value_col].values

        n = min(len(lasso_vals), len(doda_vals))
        lasso_vals, doda_vals = lasso_vals[:n], doda_vals[:n]

        if np.allclose(lasso_vals, doda_vals):
            stat, p = np.nan, 1.0
        else:
            try:
                stat, p = wilcoxon(lasso_vals, doda_vals)
            except ValueError:
                stat, p = np.nan, 1.0

        diff = doda_vals - lasso_vals
        pooled_std = np.std(np.concatenate([lasso_vals, doda_vals]), ddof=1)
        cohens_d = diff.mean() / pooled_std if pooled_std > 0 else 0.0

        rows.append({
            **(dict(zip(group_cols, keys)) if isinstance(keys, tuple) else {group_cols[0]: keys}),
            "LASSO_mean": lasso_vals.mean(),
            "DODA_mean": doda_vals.mean(),
            "p_value": p,
            "cohens_d": cohens_d
        })

    result_df = pd.DataFrame(rows)
    if len(result_df) > 0:
        reject, p_adj, _, _ = multipletests(result_df["p_value"].fillna(1.0), method="holm")
        result_df["p_holm"] = p_adj
        result_df["significant"] = reject
    return result_df


print("Helper functions defined.")

Helper functions defined.


# PRIMARY ANALYSIS (Imputed, n=400)

## 1. Baseline (80/20 split, for direct comparison with earlier notebooks)

In [8]:
# =============================================================================
# PRIMARY: TRAIN-TEST SPLIT, IMPUTATION, SCALING
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_primary, y_primary, test_size=0.2, random_state=42, stratify=y_primary
)

X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute=True)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("PRIMARY split:", X_train_scaled.shape, X_test_scaled.shape)
print("Missing after imputation:", X_train_scaled.isnull().sum().sum(),
      X_test_scaled.isnull().sum().sum())

PRIMARY split: (320, 24) (80, 24)
Missing after imputation: 0 0


In [9]:
# =============================================================================
# PRIMARY: LASSO BASELINE TOP-K
# =============================================================================

k_values = [5, 10, 15, 20]

lasso_selector = LogisticRegression(
    penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
    class_weight="balanced", random_state=42
)
lasso_selector.fit(X_train_scaled, y_train)

lasso_scores = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "LASSO Coefficient": lasso_selector.coef_[0],
    "LASSO Score": np.abs(lasso_selector.coef_[0])
}).sort_values("LASSO Score", ascending=False).reset_index(drop=True)

display(lasso_scores)

lasso_results = {}
for k in k_values:
    top_features = lasso_scores.head(k)["Feature"].tolist()
    mask = X_train_scaled.columns.isin(top_features)
    lasso_results[k] = {
        "features": top_features,
        "X_train": X_train_scaled.loc[:, mask],
        "X_test": X_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Feature,LASSO Coefficient,LASSO Score
0,hemo,-1.423788,1.423788
1,sg,-1.016467,1.016467
2,htn,0.427144,0.427144
3,dm,0.398986,0.398986
4,pcv,-0.255307,0.255307
5,al,0.251402,0.251402
6,rc,-0.145765,0.145765
7,appet,-0.107240,0.107240
8,pcc,0.000000,0.000000
9,pc,0.000000,0.000000


Top-5: ['hemo', 'sg', 'htn', 'dm', 'pcv']
Top-10: ['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'pcc', 'pc']
Top-15: ['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'pcc', 'pc', 'rbc', 'su', 'age', 'bp', 'ba']
Top-20: ['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'pcc', 'pc', 'rbc', 'su', 'age', 'bp', 'ba', 'bgr', 'sod', 'pot', 'bu', 'sc']


In [10]:
# =============================================================================
# PRIMARY: LASSO BASELINE MODEL EVALUATION
# =============================================================================

primary_baseline_results = []
for k in k_values:
    Xtr, Xts = lasso_results[k]["X_train"], lasso_results[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_baseline_results.append({
            "Method": "LASSO", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_baseline_df = pd.DataFrame(primary_baseline_results)
display(primary_baseline_df)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,LASSO,5,LR,0.9500,0.958333,0.992667
1,LASSO,5,RF,0.9750,0.979592,0.999333
2,LASSO,5,XGB,0.9875,0.990099,0.998667
3,LASSO,10,LR,0.9750,0.979592,0.999667
4,LASSO,10,RF,0.9875,0.989899,0.999667
5,LASSO,10,XGB,0.9875,0.990099,0.999667
6,LASSO,15,LR,0.9750,0.979592,0.998667
7,LASSO,15,RF,0.9875,0.989899,1.000000
8,LASSO,15,XGB,0.9875,0.990099,1.000000
9,LASSO,20,LR,0.9750,0.979592,1.000000


In [11]:
# =============================================================================
# PRIMARY: RANK FUSION DODA
# =============================================================================

provider = JSONProvider(WEIGHTS_FILE)
fusion = RankFusion()

primary_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(
        LogisticRegression(
            penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
            class_weight="balanced", random_state=42
        )
    )
    selector = DODASelector(operators=[operator], provider=provider, fusion=fusion, top_k=k)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    features = selector.get_selected_features()
    primary_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel,
                                     "features": features, "selector": selector}
    print(f"Top-{k}:", features)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b307fe60>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.0), 'bp': np.float64(0.0), 'sg': np.float64(1.0164668375534056), 'al': np.float64(0.2514024950306868), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.0), 'bu': np.float64(0.0), 'sc': np.float64(0.0), 'sod': np.float64(0.0), 'pot': np.float64(0.0), 'hemo': np.float64(1.4237881705393562), 'pcv': np.float64(0.25530704927822945), 'wc': np.float64(0.0), 'rc': np.float64(0.14576453153473526), 'htn': np.float64(0.42714351314419446), 'dm': np.float64(0.3989859776904602), 'cad': np.float64(0.0), 'appet': np.float64(0.10723998781815601), 'pe': np.float6

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

In [12]:
# =============================================================================
# PRIMARY: DODA MODEL EVALUATION + COMBINED BASELINE COMPARISON
# =============================================================================

primary_doda_results = []
for k in k_values:
    Xtr, Xts = primary_doda_results_by_k[k]["X_train"], primary_doda_results_by_k[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_doda_results.append({
            "Method": "LASSO + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_doda_df = pd.DataFrame(primary_doda_results)
primary_comparison_df = pd.concat([primary_baseline_df, primary_doda_df], ignore_index=True)
display(primary_comparison_df)

import os
os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_comparison_df.to_csv("../../../results/ckd/primary/lasso_rankfusion_80_20_comparison.csv", index=False)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,LASSO,5,LR,0.9500,0.958333,0.992667
1,LASSO,5,RF,0.9750,0.979592,0.999333
2,LASSO,5,XGB,0.9875,0.990099,0.998667
3,LASSO,10,LR,0.9750,0.979592,0.999667
4,LASSO,10,RF,0.9875,0.989899,0.999667
5,LASSO,10,XGB,0.9875,0.990099,0.999667
6,LASSO,15,LR,0.9750,0.979592,0.998667
7,LASSO,15,RF,0.9875,0.989899,1.000000
8,LASSO,15,XGB,0.9875,0.990099,1.000000
9,LASSO,20,LR,0.9750,0.979592,1.000000


## 2. Feature-Selection Stability (25-run repeated CV)

In [13]:
# =============================================================================
# PRIMARY: STABILITY ANALYSIS
# =============================================================================

primary_jaccard_df = run_stability_analysis(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_stability_summary = primary_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("PRIMARY STABILITY SUMMARY")
print("=" * 70)
display(primary_stability_summary)


PRIMARY STABILITY — TOP-5


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

LASSO: mean Jaccard = 1.0000 (std 0.0000)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d51490>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.007067988547346499), 'bp': np.float64(0.07005529207612844), 'sg': np.float64(0.0), 'al': np.float64(0.8334280024922404), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.03898959967616574), 'bu': np.float64(0.017581423357705895), 'sc': np.float64(0.5561575341140083), 'sod': np.float64(0.017603009549317646), 'pot': np.float64(0.0), 'hemo': np.float64(0.5218637312485643), 'pcv': np.float64(0.18729707546777893), 'wc': np.float64(0.00011666575812820382), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': n

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

...

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1df7dd0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.007830315168854013), 'bp': np.float64(0.06790491469519082), 'sg': np.float64(0.0), 'al': np.float64(0.8579071586810849), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.037562566771150524), 'bu': np.float64(0.013558496190445371), 'sc': np.float64(0.6564004579921074), 'sod': np.float64(0.025884245527680188), 'pot': np.float64(0.0), 'hemo': np.float64(0.556002889293745), 'pcv': np.float64(0.18892743777504917), 'wc': np.float64(8.991942033845394e-05), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'ap

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.0005381255531991614), 'bp': np.float64(0.06427054495774148), 'sg': np.float64(0.0), 'al': np.float64(0.9348368858424752), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.03557969603611913), 'bu': np.float64(0.021561193526391742), 'sc': np.float64(0.5509972714467956), 'sod': np.float64(0.02697409290477769), 'pot': np.float64(0.0), 'hemo': np.float64(0.633690106448144), 'pcv': np.float64(0.17806528829859214), 'wc': np.float64(0.0001387496690939887), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float64(0.0)}}
Resolving scores from: 1 operators

Raw Mathematical Scores
{'age': np.float64(0.000538125553

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

,Top_K,Method,mean,std
0,5,DODA,1.000000,0.000000
1,5,LASSO,1.000000,0.000000
2,10,DODA,0.985455,0.049408
3,10,LASSO,0.939394,0.085853
4,15,DODA,1.000000,0.000000
5,15,LASSO,1.000000,0.000000
6,20,DODA,1.000000,0.000000
7,20,LASSO,1.000000,0.000000


In [14]:
# =============================================================================
# PRIMARY: STABILITY SIGNIFICANCE TESTING (Wilcoxon + Holm + Cohen's d)
# =============================================================================

primary_stability_test = wilcoxon_holm_test(primary_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("PRIMARY — LASSO vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_stability_test.round(4))

PRIMARY — LASSO vs DODA STABILITY: SIGNIFICANCE TEST


,Top_K,LASSO_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,1.0000,1.0000,1.0,0.0000,1.0,False
1,10,0.9394,0.9855,0.0,0.6251,0.0,True
2,15,1.0000,1.0000,1.0,0.0000,1.0,False
3,20,1.0000,1.0000,1.0,0.0000,1.0,False


## 3. Predictive Performance (25-run repeated CV)

In [15]:
# =============================================================================
# PRIMARY: CV PREDICTIVE PERFORMANCE
# =============================================================================

primary_cv_results = run_cv_performance(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_cv_summary = primary_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
primary_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                               "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_cv_results.to_csv("../../../results/ckd/primary/cv_performance_runs.csv", index=False)
primary_cv_summary.to_csv("../../../results/ckd/primary/cv_performance_summary.csv", index=False)

print("=" * 70)
print("PRIMARY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(primary_cv_summary.round(4))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d63e90>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.007067988547346499), 'bp': np.float64(0.07005529207612844), 'sg': np.float64(0.0), 'al': np.float64(0.8334280024922404), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.03898959967616574), 'bu': np.float64(0.017581423357705895), 'sc': np.float64(0.5561575341140083), 'sod': np.float64(0.017603009549317646), 'pot': np.float64(0.0), 'hemo': np.float64(0.5218637312485643), 'pcv': np.float64(0.18729707546777893), 'wc': np.float64(0.00011666575812820382), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'a

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d50aa0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.007067988547346499), 'bp': np.float64(0.07005529207612844), 'sg': np.float64(0.0), 'al': np.float64(0.8334280024922404), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.03898959967616574), 'bu': np.float64(0.017581423357705895), 'sc': np.float64(0.5561575341140083), 'sod': np.float64(0.017603009549317646), 'pot': np.float64(0.0), 'hemo': np.float64(0.5218637312485643), 'pcv': np.float64(0.18729707546777893), 'wc': np.float64(0.00011666575812820382), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'a

...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6a9566e70>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.004250014798137275), 'bp': np.float64(0.06979520204222205), 'sg': np.float64(0.0), 'al': np.float64(0.3614753550709248), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.04009927720732392), 'bu': np.float64(0.0175200305555096), 'sc': np.float64(0.7440666371702123), 'sod': np.float64(0.029126553674230803), 'pot': np.float64(0.0), 'hemo': np.float64(0.6280817291596367), 'pcv': np.float64(0.18980986117724583), 'wc': np.float64(6.406460399427564e-05), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appe

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


PRIMARY: completed 25/25 CV runs
PRIMARY CV PERFORMANCE SUMMARY


,Top_K,Method,Model,Acc_Mean,Acc_STD,F1_Mean,F1_STD,AUC_Mean,AUC_STD
0,5,DODA,LR,0.9555,0.0242,0.9636,0.0204,0.9892,0.0115
1,5,DODA,RF,0.9680,0.0201,0.9739,0.0168,0.9903,0.0093
2,5,DODA,XGB,0.9615,0.0236,0.9687,0.0197,0.9898,0.0094
3,5,LASSO,LR,0.9555,0.0242,0.9636,0.0204,0.9892,0.0115
4,5,LASSO,RF,0.9685,0.0184,0.9743,0.0154,0.9904,0.0093
5,5,LASSO,XGB,0.9610,0.0229,0.9683,0.0192,0.9899,0.0094
6,10,DODA,LR,0.9665,0.0168,0.9727,0.0139,0.9943,0.0068
7,10,DODA,RF,0.9840,0.0151,0.9872,0.0120,0.9989,0.0011
8,10,DODA,XGB,0.9745,0.0199,0.9795,0.0162,0.9978,0.0024
9,10,LASSO,LR,0.9575,0.0260,0.9651,0.0220,0.9929,0.0089


In [16]:
# =============================================================================
# PRIMARY: PERFORMANCE SIGNIFICANCE TESTING
# =============================================================================

primary_perf_test = wilcoxon_holm_test(
    primary_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("PRIMARY — LASSO vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_perf_test.round(4))

PRIMARY — LASSO vs DODA ROC-AUC: SIGNIFICANCE TEST


,Top_K,Model,LASSO_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,LR,0.9892,0.9892,1.0000,0.0000,1.0000,False
1,5,RF,0.9904,0.9903,0.6229,-0.0116,1.0000,False
2,5,XGB,0.9899,0.9898,0.6147,-0.0072,1.0000,False
3,10,LR,0.9929,0.9943,0.0130,0.1764,0.1296,False
4,10,RF,0.9986,0.9989,0.0387,0.2619,0.3096,False
5,10,XGB,0.9977,0.9978,0.8739,0.0558,1.0000,False
6,15,LR,0.9991,0.9959,0.0013,-0.7028,0.0161,True
7,15,RF,0.9997,0.9993,0.0338,-0.5632,0.3038,False
8,15,XGB,0.9988,0.9977,0.0033,-0.4874,0.0359,True
9,20,LR,0.9997,0.9999,0.0702,0.4255,0.4592,False


# SENSITIVITY ANALYSIS (Complete-Case, n=158)

Identical pipeline, applied to the 158 rows with zero missing values — no imputation step at all. If the Primary and Sensitivity conclusions agree, the imputation choice wasn't driving the results.

## 1. Baseline (80/20 split)

In [17]:

# =============================================================================
# SENSITIVITY: TRAIN-TEST SPLIT + SCALING (no imputation needed)
# =============================================================================

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_sensitivity, y_sensitivity, test_size=0.2, random_state=42, stratify=y_sensitivity
)

scaler_s = StandardScaler()
Xs_train_scaled = pd.DataFrame(scaler_s.fit_transform(Xs_train), columns=Xs_train.columns)
Xs_test_scaled = pd.DataFrame(scaler_s.transform(Xs_test), columns=Xs_test.columns)
ys_train = ys_train.reset_index(drop=True)
ys_test = ys_test.reset_index(drop=True)

print("SENSITIVITY split:", Xs_train_scaled.shape, Xs_test_scaled.shape)

SENSITIVITY split: (126, 24) (32, 24)


In [18]:
# =============================================================================
# SENSITIVITY: LASSO BASELINE TOP-K
# =============================================================================

lasso_selector_s = LogisticRegression(
    penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
    class_weight="balanced", random_state=42
)
lasso_selector_s.fit(Xs_train_scaled, ys_train)

lasso_scores_s = pd.DataFrame({
    "Feature": Xs_train_scaled.columns,
    "LASSO Coefficient": lasso_selector_s.coef_[0],
    "LASSO Score": np.abs(lasso_selector_s.coef_[0])
}).sort_values("LASSO Score", ascending=False).reset_index(drop=True)

display(lasso_scores_s)

lasso_results_s = {}
for k in k_values:
    top_features = lasso_scores_s.head(k)["Feature"].tolist()
    mask = Xs_train_scaled.columns.isin(top_features)
    lasso_results_s[k] = {
        "features": top_features,
        "X_train": Xs_train_scaled.loc[:, mask],
        "X_test": Xs_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Feature,LASSO Coefficient,LASSO Score
0,al,1.750828,1.750828
1,sg,-0.229718,0.229718
2,htn,0.226350,0.226350
3,hemo,-0.116927,0.116927
4,pcv,-0.112393,0.112393
5,su,0.000000,0.000000
6,bp,0.000000,0.000000
7,age,0.000000,0.000000
8,pcc,0.000000,0.000000
9,pc,0.000000,0.000000


Top-5: ['al', 'sg', 'htn', 'hemo', 'pcv']
Top-10: ['al', 'sg', 'htn', 'hemo', 'pcv', 'su', 'bp', 'age', 'pcc', 'pc']
Top-15: ['al', 'sg', 'htn', 'hemo', 'pcv', 'su', 'bp', 'age', 'pcc', 'pc', 'rbc', 'ba', 'sc', 'bu', 'pot']
Top-20: ['al', 'sg', 'htn', 'hemo', 'pcv', 'su', 'bp', 'age', 'pcc', 'pc', 'rbc', 'ba', 'sc', 'bu', 'pot', 'bgr', 'sod', 'wc', 'rc', 'dm']


In [19]:
# =============================================================================
# SENSITIVITY: LASSO BASELINE MODEL EVALUATION
# =============================================================================

sensitivity_baseline_results = []
for k in k_values:
    Xtr, Xts = lasso_results_s[k]["X_train"], lasso_results_s[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_baseline_results.append({
            "Method": "LASSO", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_baseline_df = pd.DataFrame(sensitivity_baseline_results)
display(sensitivity_baseline_df)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,LASSO,5,LR,1.00000,1.000000,1.0
1,LASSO,5,RF,1.00000,1.000000,1.0
2,LASSO,5,XGB,0.96875,0.941176,1.0
3,LASSO,10,LR,1.00000,1.000000,1.0
4,LASSO,10,RF,1.00000,1.000000,1.0
5,LASSO,10,XGB,0.96875,0.941176,1.0
6,LASSO,15,LR,1.00000,1.000000,1.0
7,LASSO,15,RF,1.00000,1.000000,1.0
8,LASSO,15,XGB,0.96875,0.941176,1.0
9,LASSO,20,LR,1.00000,1.000000,1.0


In [20]:
# =============================================================================
# SENSITIVITY: RANK FUSION DODA + EVALUATION
# =============================================================================

sensitivity_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(
        LogisticRegression(
            penalty="l1", solver="liblinear", C=0.1, max_iter=2000,
            class_weight="balanced", random_state=42
        )
    )
    selector = DODASelector(operators=[operator], provider=provider, fusion=RankFusion(), top_k=k)
    X_train_sel = selector.fit_transform(Xs_train_scaled, ys_train)
    X_test_sel = selector.transform(Xs_test_scaled)
    features = selector.get_selected_features()
    sensitivity_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel, "features": features}
    print(f"Top-{k}:", features)

sensitivity_doda_results = []
for k in k_values:
    Xtr, Xts = sensitivity_doda_results_by_k[k]["X_train"], sensitivity_doda_results_by_k[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_doda_results.append({
            "Method": "LASSO + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_doda_df = pd.DataFrame(sensitivity_doda_results)
sensitivity_comparison_df = pd.concat([sensitivity_baseline_df, sensitivity_doda_df], ignore_index=True)
display(sensitivity_comparison_df)

os.makedirs("../../../results/ckd/sensitivity", exist_ok=True)
sensitivity_comparison_df.to_csv("../../../results/ckd/sensitivity/lasso_rankfusion_80_20_comparison.csv", index=False)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b307fe60>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.0), 'bp': np.float64(0.0), 'sg': np.float64(0.22971809246401917), 'al': np.float64(1.7508281929454714), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.0), 'bu': np.float64(0.0), 'sc': np.float64(0.0), 'sod': np.float64(0.0), 'pot': np.float64(0.0), 'hemo': np.float64(0.11692668238132264), 'pcv': np.float64(0.11239342723730132), 'wc': np.float64(0.0), 'rc': np.float64(0.0), 'htn': np.float64(0.22634985896113932), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float64(0.0)}}
Resolving sc

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,LASSO,5,LR,1.00000,1.000000,1.0
1,LASSO,5,RF,1.00000,1.000000,1.0
2,LASSO,5,XGB,0.96875,0.941176,1.0
3,LASSO,10,LR,1.00000,1.000000,1.0
4,LASSO,10,RF,1.00000,1.000000,1.0
5,LASSO,10,XGB,0.96875,0.941176,1.0
6,LASSO,15,LR,1.00000,1.000000,1.0
7,LASSO,15,RF,1.00000,1.000000,1.0
8,LASSO,15,XGB,0.96875,0.941176,1.0
9,LASSO,20,LR,1.00000,1.000000,1.0


## 2. Feature-Selection Stability (25-run repeated CV)

In [21]:
# =============================================================================
# SENSITIVITY: STABILITY ANALYSIS
# =============================================================================

sensitivity_jaccard_df = run_stability_analysis(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_stability_summary = sensitivity_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("SENSITIVITY STABILITY SUMMARY")
print("=" * 70)
display(sensitivity_stability_summary)


SENSITIVITY STABILITY — TOP-5
LASSO: mean Jaccard = 0.8844 (std 0.1586)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d34a40>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.019789925502335427), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.06327604596869646), 'bu': np.float64(0.07898341148757096), 'sc': np.float64(0.0), 'sod': np.float64(0.054564148028266916), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.24904010277258315), 'wc': np.float64(0.0005038247590906192), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float6

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

LASSO: mean Jaccard = 1.0000 (std 0.0000)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d4b8f0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.019789925502335427), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.06327604596869646), 'bu': np.float64(0.07898341148757096), 'sc': np.float64(0.0), 'sod': np.float64(0.054564148028266916), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.24904010277258315), 'wc': np.float64(0.0005038247590906192), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d53bf0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.030108391364067656), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.056801605936279996), 'bu': np.float64(0.06396352007282712), 'sc': np.float64(0.0), 'sod': np.float64(0.05758364180947746), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.23394781325581473), 'wc': np.float64(0.0005508537556352861), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float6

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6a94b6060>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.0), 'bp': np.float64(0.0032568689945717162), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.04768662392013308), 'bu': np.float64(0.12380756758670505), 'sc': np.float64(0.0), 'sod': np.float64(0.13966373051120684), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.08828027236132387), 'wc': np.float64(0.001043994423412716), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float64

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/u

,Top_K,Method,mean,std
0,5,DODA,0.813333,0.165739
1,5,LASSO,0.884444,0.158901
2,10,DODA,0.939394,0.085853
3,10,LASSO,1.000000,0.000000
4,15,DODA,1.000000,0.000000
5,15,LASSO,1.000000,0.000000
6,20,DODA,0.992381,0.025881
7,20,LASSO,1.000000,0.000000


In [22]:
sensitivity_stability_test = wilcoxon_holm_test(sensitivity_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("SENSITIVITY — LASSO vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_stability_test.round(4))

SENSITIVITY — LASSO vs DODA STABILITY: SIGNIFICANCE TEST


,Top_K,LASSO_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,0.8844,0.8133,0.0,-0.4282,0.0,True
1,10,1.0000,0.9394,0.0,-0.8937,0.0,True
2,15,1.0000,1.0000,1.0,0.0000,1.0,False
3,20,1.0000,0.9924,0.0,-0.4079,0.0,True


## 3. Predictive Performance (25-run repeated CV)

In [23]:
# =============================================================================
# SENSITIVITY: CV PREDICTIVE PERFORMANCE
# =============================================================================

sensitivity_cv_results = run_cv_performance(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_cv_summary = sensitivity_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
sensitivity_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                                   "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

sensitivity_cv_results.to_csv("../../../results/ckd/sensitivity/cv_performance_runs.csv", index=False)
sensitivity_cv_summary.to_csv("../../../results/ckd/sensitivity/cv_performance_summary.csv", index=False)

print("=" * 70)
print("SENSITIVITY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(sensitivity_cv_summary.round(4))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d68050>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.019789925502335427), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.06327604596869646), 'bu': np.float64(0.07898341148757096), 'sc': np.float64(0.0), 'sod': np.float64(0.054564148028266916), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.24904010277258315), 'wc': np.float64(0.0005038247590906192), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float6

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b1d52060>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.019789925502335427), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.06327604596869646), 'bu': np.float64(0.07898341148757096), 'sc': np.float64(0.0), 'sod': np.float64(0.054564148028266916), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.24904010277258315), 'wc': np.float64(0.0005038247590906192), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float6

...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7fa6b2471be0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.03427300210165808), 'bp': np.float64(0.0), 'sg': np.float64(0.0), 'al': np.float64(0.0), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.06170216637965036), 'bu': np.float64(0.07563905141140272), 'sc': np.float64(0.0), 'sod': np.float64(0.05686936865994881), 'pot': np.float64(0.0), 'hemo': np.float64(0.0), 'pcv': np.float64(0.24734109272138616), 'wc': np.float64(0.0004717608471438605), 'rc': np.float64(0.0), 'htn': np.float64(0.0), 'dm': np.float64(0.0), 'cad': np.float64(0.0), 'appet': np.float64(0.0), 'pe': np.float64(0.0), 'ane': np.float64(

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


SENSITIVITY: completed 25/25 CV runs
SENSITIVITY CV PERFORMANCE SUMMARY


,Top_K,Method,Model,Acc_Mean,Acc_STD,F1_Mean,F1_STD,AUC_Mean,AUC_STD
0,5,DODA,LR,0.9949,0.0119,0.9900,0.0235,1.0000,0.0000
1,5,DODA,RF,0.9937,0.0129,0.9876,0.0254,1.0000,0.0000
2,5,DODA,XGB,0.9911,0.0145,0.9826,0.0286,1.0000,0.0000
3,5,LASSO,LR,0.9822,0.0161,0.9645,0.0322,0.9925,0.0141
4,5,LASSO,RF,0.9810,0.0159,0.9625,0.0314,1.0000,0.0000
5,5,LASSO,XGB,0.9936,0.0130,0.9873,0.0260,1.0000,0.0000
6,10,DODA,LR,0.9950,0.0118,0.9903,0.0228,1.0000,0.0000
7,10,DODA,RF,0.9937,0.0129,0.9876,0.0254,1.0000,0.0000
8,10,DODA,XGB,0.9898,0.0151,0.9799,0.0300,1.0000,0.0000
9,10,LASSO,LR,0.9872,0.0160,0.9743,0.0322,1.0000,0.0000


In [24]:
sensitivity_perf_test = wilcoxon_holm_test(
    sensitivity_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("SENSITIVITY — LASSO vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_perf_test.round(4))

SENSITIVITY — LASSO vs DODA ROC-AUC: SIGNIFICANCE TEST


,Top_K,Model,LASSO_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,LR,0.9925,1.0000,0.0277,0.7044,0.3325,False
1,5,RF,1.0000,1.0000,1.0000,0.0000,1.0000,False
2,5,XGB,1.0000,1.0000,1.0000,-0.2800,1.0000,False
3,10,LR,1.0000,1.0000,1.0000,0.0000,1.0000,False
4,10,RF,1.0000,1.0000,1.0000,0.0000,1.0000,False
5,10,XGB,1.0000,1.0000,1.0000,-0.2800,1.0000,False
6,15,LR,1.0000,0.9983,0.3173,-0.2828,1.0000,False
7,15,RF,1.0000,1.0000,1.0000,0.0000,1.0000,False
8,15,XGB,1.0000,0.9998,0.6547,-0.2828,1.0000,False
9,20,LR,1.0000,1.0000,1.0000,0.0000,1.0000,False


# Primary vs. Sensitivity — Head-to-Head Comparison

In [25]:
# =============================================================================
# SIDE-BY-SIDE: STABILITY SUMMARY
# =============================================================================

primary_stability_summary["Analysis"] = "Primary (imputed, n=400)"
sensitivity_stability_summary["Analysis"] = "Sensitivity (complete-case, n=158)"

stability_side_by_side = pd.concat(
    [primary_stability_summary, sensitivity_stability_summary], ignore_index=True
).pivot_table(index=["Top_K", "Method"], columns="Analysis", values="mean").round(4)

print("=" * 70)
print("MEAN JACCARD STABILITY — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(stability_side_by_side)

MEAN JACCARD STABILITY — PRIMARY vs SENSITIVITY


Analysis      Primary (imputed, n=400)  Sensitivity (complete-case, n=158)
Top_K Method                                                              
5     DODA                      1.0000                              0.8133
      LASSO                     1.0000                              0.8844
10    DODA                      0.9855                              0.9394
      LASSO                     0.9394                              1.0000
15    DODA                      1.0000                              1.0000
      LASSO                     1.0000                              1.0000
20    DODA                      1.0000                              0.9924
      LASSO                     1.0000                              1.0000

In [26]:
# =============================================================================
# SIDE-BY-SIDE: PREDICTIVE PERFORMANCE (ROC-AUC) SUMMARY
# =============================================================================

primary_auc = primary_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
primary_auc["Analysis"] = "Primary"
sensitivity_auc = sensitivity_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
sensitivity_auc["Analysis"] = "Sensitivity"

auc_side_by_side = pd.concat([primary_auc, sensitivity_auc], ignore_index=True).pivot_table(
    index=["Top_K", "Method", "Model"], columns="Analysis", values="AUC_Mean"
).round(4)

print("=" * 70)
print("MEAN ROC-AUC — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(auc_side_by_side)

auc_side_by_side.to_csv("../../../results/ckd/lasso_primary_vs_sensitivity_auc_comparison.csv")
stability_side_by_side.to_csv("../../../results/ckd/lasso_primary_vs_sensitivity_stability_comparison.csv")

MEAN ROC-AUC — PRIMARY vs SENSITIVITY


Analysis            Primary  Sensitivity
Top_K Method Model                      
5     DODA   LR      0.9892       1.0000
             RF      0.9903       1.0000
             XGB     0.9898       1.0000
      LASSO  LR      0.9892       0.9925
             RF      0.9904       1.0000
             XGB     0.9899       1.0000
10    DODA   LR      0.9943       1.0000
             RF      0.9989       1.0000
             XGB     0.9978       1.0000
      LASSO  LR      0.9929       1.0000
             RF      0.9986       1.0000
             XGB     0.9977       1.0000
15    DODA   LR      0.9959       0.9983
             RF      0.9993       1.0000
             XGB     0.9977       0.9998
      LASSO  LR      0.9991       1.0000
             RF      0.9997       1.0000
             XGB     0.9988       1.0000
20    DODA   LR      0.9999       1.0000
             RF      0.9999       1.0000
             XGB     0.9989       1.0000
      LASSO  LR      0.9997       1.0000
             RF      0.9998       1.0000
             XGB     0.9988       0.9998